In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pubplots as pp
from matplotlib.colors import to_rgba
from scipy.ndimage import gaussian_filter

from cnpix_local_sleep import sps_conf
from cnpix_local_sleep.morphological import correlation_stats
from cnpix_local_sleep.morphological.mua import files as mua_files
from cnpix_local_sleep import plots
from cnpix_local_sleep.morphological.pipeline import add_bandpower_to_offs
from cnpix_local_sleep.morphological.pipeline.postprocess_offs import postprocess_offs_frame
from cnpix_local_sleep import off_tables


In [ ]:
save_plots = True
plot_for_poster = True

# Publication-figure styling for the LLAS scatter grid (see the scatter cell).
# When True: the per-panel rho annotation drops the "small/moderate/large"
# effect-size word, and the y-axis (bandpower) labels are omitted so the bare
# SVG can be labelled downstream (e.g. in Figma).
PLOT_FOR_PUB = True

# This notebook deals only with whole-recording morphological OFFs
# (mua.files.get_full_offs_path). Every OFF is classified by its per-OFF
# `state`, and the analysis uses all NREM OFFs and all Wake OFFs. These OFFs are
# never subset to the experimental condition windows (e.g. Wake OFFs are not
# restricted to Early.NOD.Wake + Late.NOD.Wake).

# Recompute (instead of reusing the cached) assembled full-48h OFF table.
# Assembling it is heavy (per-structure bandpower I/O), so it is cached.
REBUILD_FULL48H_CACHE = False

# --- Category-selection nesting --------------------------------------------
# Each OFF is labeled by the most restrictive nested OFF-property filter tier it
# passes (LLAS >= CLAS >= BLAS; cnpix_local_sleep.off_tables). A "selection" maps to
# the set of those `category` labels it includes:
#   * "LLAS-exclusive": OFFs passing only the LLAS filters     (category == "LLAS")
#   * "CLAS-exclusive": OFFs passing CLAS but not BLAS          (category == "CLAS")
#   * "BLAS-exclusive": OFFs passing the BLAS filters           (category == "BLAS")
#   * "LLAS":           all OFFs (LLAS, including CLAS and BLAS)
#   * "CLAS":           CLAS, including BLAS
CATEGORY_SELECTION_MEMBERS = {
    "LLAS-exclusive": ["LLAS"],
    "CLAS-exclusive": ["CLAS"],
    "BLAS-exclusive": ["BLAS"],
    "LLAS": ["LLAS", "CLAS", "BLAS"],
    "CLAS": ["CLAS", "BLAS"],
}

# User-selected list: which selections to produce each family of plots for.
# Edit this list to control which selections are plotted.
CATEGORY_SELECTIONS = [
    "LLAS-exclusive",
    "CLAS-exclusive",
    "BLAS-exclusive",
    "LLAS",
    "CLAS",
]

_unknown = [s for s in CATEGORY_SELECTIONS if s not in CATEGORY_SELECTION_MEMBERS]
if _unknown:
    raise ValueError(
        f"Unknown CATEGORY_SELECTIONS {_unknown}; "
        f"valid: {sorted(CATEGORY_SELECTION_MEMBERS)}"
    )


In [ ]:
OUTPUT_DIR = pathlib.Path("./outputs/has_value")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)


In [ ]:
merge_keys = ["subject", "probe", "structure", "start_time", "end_time"]

_CACHE_DIR = pathlib.Path("./outputs/bandpower_vs_off")
_CACHE_DIR.mkdir(parents=True, exist_ok=True)
FULL48H_CACHE_PATH = _CACHE_DIR / "full48h_morphological_offs_with_bandpower.parquet"


def _load_full48h_morphological_offs():
    """Assemble whole-recording morphological cortical OFFs with bandpower.

    Reads every cortical (subject, probe, structure) full-recording
    ``offs.parquet`` (``mua.files.get_full_offs_path``), adds the canonical
    postprocessing columns (``postprocess_offs_frame``), and attaches the same
    bandpower columns the LLAS/CLAS/BLAS pipeline uses
    (``add_bandpower_to_offs.add_bandpower_columns``). The per-OFF ``state``
    column is preserved so NREM/Wake can be classified directly. Cached to
    parquet because the per-structure bandpower I/O is heavy.
    """
    if FULL48H_CACHE_PATH.exists() and not REBUILD_FULL48H_CACHE:
        print(f"Loading cached full-48h OFFs from {FULL48H_CACHE_PATH}")
        return pd.read_parquet(FULL48H_CACHE_PATH)

    spsl = sps_conf.get_subject_probe_structure_list(
        method="morphological",
        exclude_thalamus=True,
        exclude_striatum=True,
        exclude_other=True,
    )
    frames = []
    for subject, probe, structure in spsl:
        fpath = mua_files.get_full_offs_path(subject, probe, structure)
        if not fpath.exists():
            print(f"  missing full-48h offs, skipping: {subject} {probe} {structure}")
            continue
        o = pd.read_parquet(fpath)
        if o.empty:
            continue
        o["subject"] = subject
        o["probe"] = probe
        o["structure"] = structure
        postprocess_offs_frame(o, structure)
        frames.append(o)

    if not frames:
        raise RuntimeError(
            "No full-48h morphological offs.parquet files found. "
            "Run `morphological-offs detect-offs-full` first."
        )

    offs = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(offs)} raw full-48h OFFs; attaching bandpower (heavy)...")
    offs = add_bandpower_to_offs.add_bandpower_columns(offs).reset_index(drop=True)
    offs.to_parquet(FULL48H_CACHE_PATH)
    print(f"Cached assembled OFFs to {FULL48H_CACHE_PATH}")
    return offs


def _assign_filter_category(offs):
    """Label each OFF by the most restrictive nested filter tier it passes.

    Mirrors the LLAS >= CLAS >= BLAS nesting used by the aggregation pipeline
    (``cnpix_local_sleep.off_tables`` filters): an OFF is "BLAS" if it passes the
    BLAS filters, else "CLAS", else "LLAS". OFFs failing the loosest (LLAS)
    filters are dropped, matching the per-condition parquets' LLAS superset.
    """

    def _passes(filters):
        mask = pd.Series(True, index=offs.index)
        for col, (lo, hi) in filters.items():
            mask &= offs[col].between(lo, hi)
        return mask

    llas_m = _passes(off_tables.llas_filters)
    clas_m = _passes(off_tables.clas_filters)
    blas_m = _passes(off_tables.blas_filters)

    offs = offs.loc[llas_m].copy()
    category = np.where(
        blas_m.loc[offs.index],
        "BLAS",
        np.where(clas_m.loc[offs.index], "CLAS", "LLAS"),
    )
    offs["category"] = pd.Categorical(
        category, categories=["LLAS", "CLAS", "BLAS"], ordered=True
    )
    return offs.reset_index(drop=True)


def _classify_state(offs):
    """Map each OFF to "NREM", "Wake", or NA via the per-OFF ``state`` column.

    Whole-recording morphological OFFs carry a ``state`` label, so NREM and Wake are
    true whole-recording populations (never subset to condition windows). OFFs in
    other states (IS/REM/MA/...) map to NA and are dropped downstream.
    """
    state_class = pd.Series(pd.NA, index=offs.index, dtype="object")
    state = offs["state"].astype("string")
    state_class[state == "NREM"] = "NREM"
    state_class[state == "Wake"] = "Wake"
    return state_class


offs = _assign_filter_category(_load_full48h_morphological_offs())

# Classify NREM vs Wake and keep only those OFFs: this analysis compares the
# NREM and Wake populations, so OFFs in other states are excluded.
offs["state_class"] = _classify_state(offs)
offs = offs[offs["state_class"].notna()].reset_index(drop=True)
print(
    f"{len(offs)} morphological OFFs"
    f"\n  state_class: {offs['state_class'].value_counts().to_dict()}"
    f"\n  category:    {offs['category'].value_counts().to_dict()}"
)

# Identify available OFF-property and bandpower columns
_bandpower_prefixes = ("total_", "mean_", "median_", "max_")
_bandpower_suffixes = ("_delta", "_eta")
bandpower_cols = sorted(
    c
    for c in offs.columns
    if any(c.startswith(p) for p in _bandpower_prefixes)
    and any(c.endswith(s) for s in _bandpower_suffixes)
)
off_property_cols = sorted(
    c for c in offs.select_dtypes(include="number").columns if c not in bandpower_cols
)
print(f"\nOFF property columns ({len(off_property_cols)}):\n  {off_property_cols}")
print(f"\nBandpower columns ({len(bandpower_cols)}):\n  {bandpower_cols}")


In [ ]:
if plot_for_poster:
    subject_map = {
        s: f"Subject{i}"
        for i, s in enumerate(sorted(offs["subject"].unique()), start=1)
    }


In [ ]:
def _filter_by_condition(offs, condition_filter):
    """Subset OFFs to a sleep/wake population via the ``state_class`` column.

    ``"NREM"`` / ``"Wake"`` select OFFs whose ``state_class`` matches; ``"all"``
    keeps every OFF (the NREM union Wake population the loader already restricted
    to). ``state_class`` is derived from each OFF's whole-recording ``state``.
    """
    if condition_filter == "all":
        return offs
    if condition_filter in ("NREM", "Wake"):
        return offs[offs["state_class"] == condition_filter]
    raise ValueError(
        f"condition_filter must be 'all', 'NREM', or 'Wake', got {condition_filter!r}"
    )


def _filter_by_category_selection(offs, selection):
    """Subset OFFs to a category selection's nested-filter membership.

    ``selection`` keys ``CATEGORY_SELECTION_MEMBERS`` (e.g. "BLAS-exclusive",
    "LLAS"); OFFs whose ``category`` is in that selection's member list are kept.
    The nesting LLAS >= CLAS >= BLAS means e.g. "CLAS" keeps both CLAS and BLAS
    OFFs, while "CLAS-exclusive" keeps only OFFs that fail the tighter BLAS tier.
    """
    members = CATEGORY_SELECTION_MEMBERS[selection]
    return offs[offs["category"].isin(members)]


def _cooks_distance(x, y):
    """Cook's distance per point for a simple OLS fit of ``y`` on ``x``.

    For a simple linear regression (``p = 2`` parameters) Cook's distance is

        D_i = e_i**2 / (p * MSE) * h_i / (1 - h_i)**2

    where ``e_i`` is the raw residual, ``MSE = sum(e**2) / (n - p)``, and
    ``h_i = 1/n + (x_i - xbar)**2 / Sxx`` is the hat-value leverage. Returns an
    all-zeros array (nothing influential) when the fit is degenerate -- too few
    points, zero ``x``-variance, or zero residual variance. Maximal-leverage
    points (``h_i == 1``) get ``inf``.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    p = 2  # slope + intercept
    if n <= p:
        return np.zeros(n)
    xbar = x.mean()
    sxx = np.sum((x - xbar) ** 2)
    if sxx == 0:
        return np.zeros(n)

    slope, intercept = np.polyfit(x, y, deg=1)
    resid = y - (slope * x + intercept)
    mse = np.sum(resid**2) / (n - p)
    if mse == 0:
        return np.zeros(n)

    h = 1.0 / n + (x - xbar) ** 2 / sxx
    denom = (1.0 - h) ** 2
    with np.errstate(divide="ignore", invalid="ignore"):
        cooks = (resid**2 / (p * mse)) * (h / denom)
    return np.where(denom > 0, cooks, np.inf)


def _influence_keep(x, y, percentile=0.995):
    """Keep-mask dropping the most influential points by Cook's distance.

    Excludes points whose Cook's distance exceeds the ``percentile`` quantile of
    the group's own D distribution (default 0.995 -> drop the top 0.5%). Unlike a
    fixed ``D`` cutoff, this trims a predictable, small fraction regardless of
    group size -- appropriate here, where per-group n is large and OFF
    properties are heavy-tailed.
    """
    cooks = _cooks_distance(x, y)
    n = len(cooks)
    if n <= 2:
        return np.ones(n, dtype=bool)
    cutoff = np.quantile(cooks, percentile)
    return cooks <= cutoff


def _anonymize_gc(gc, subject_map):
    """Anonymize a group-correlations DataFrame for poster display."""
    gc = gc.copy()
    gc["subject"] = gc["subject"].map(subject_map)
    gc = gc.drop(columns="probe", errors="ignore")
    return gc


def _category_density_contours(
    ax,
    sub,
    x_col,
    y_col,
    colors,
    levels=4,
    gridsize=120,
    smoothing=3.0,
    fill=False,
    linewidths=1.0,
    level_min=0.1,
    level_max=0.9,
    log_levels=False,
    category_order=("LLAS", "CLAS", "BLAS"),
):
    """Overlay a fast per-category 2D density estimate as contours.

    For each category in ``category_order``, bins its (x, y) points into a
    ``gridsize`` x ``gridsize`` 2D histogram over the shared data range,
    Gaussian-smooths it (``smoothing`` = sigma in bins, i.e. the KDE bandwidth),
    normalizes to the category's own peak, and draws ``levels`` contour lines (or
    filled bands, ``fill=True``) in the category's palette color. A
    binned-and-smoothed estimate keeps this responsive at the full per-recording
    N -- binning is O(N), independent of the contour-grid resolution, so no
    subsampling is needed. Per-category peak-normalization makes every category's
    shape visible regardless of how many points it holds.

    The contour elevations run from ``level_min`` to ``level_max`` as fractions
    of each category's peak density. ``level_min`` is the crucial knob for
    categories with a very peaked mode plus a diffuse shoulder (e.g. a merged
    CLAS+BLAS whose density spikes on the tightly-clustered BLAS mode): a low
    diffuse region only gets a contour if ``level_min`` is dropped below its
    peak-relative height. ``log_levels`` spaces the levels geometrically
    (``np.geomspace``) instead of linearly, distributing rings across the
    density's dynamic range instead of bunching them near the peak (``level_min``
    must be > 0).
    """
    xy = sub[[x_col, y_col]].dropna()
    if len(xy) < 10:
        return
    xedges = np.linspace(xy[x_col].min(), xy[x_col].max(), gridsize + 1)
    yedges = np.linspace(xy[y_col].min(), xy[y_col].max(), gridsize + 1)
    xcent = 0.5 * (xedges[:-1] + xedges[1:])
    ycent = 0.5 * (yedges[:-1] + yedges[1:])
    if log_levels:
        contour_levels = np.geomspace(level_min, level_max, levels)
    else:
        contour_levels = np.linspace(level_min, level_max, levels)
    for cat in category_order:
        cs = sub.loc[sub["category"] == cat, [x_col, y_col]].dropna()
        if len(cs) < 10:
            continue
        hist, _, _ = np.histogram2d(
            cs[x_col].values, cs[y_col].values, bins=[xedges, yedges]
        )
        hist = gaussian_filter(hist, sigma=smoothing)
        peak = hist.max()
        if peak <= 0:
            continue
        density = (hist / peak).T  # transpose: contour expects Z indexed [y, x]
        if fill:
            ax.contourf(
                xcent,
                ycent,
                density,
                levels=contour_levels,
                colors=colors[cat],
                alpha=0.3,
            )
        else:
            ax.contour(
                xcent,
                ycent,
                density,
                levels=contour_levels,
                colors=colors[cat],
                linewidths=pp.scale(linewidths),
            )


def _draw_recording_scatter(
    ax,
    sub,
    x_col,
    y_col,
    colors,
    show_regline=False,
    cooks_percentile=0.999,
    log_x=False,
    gc_row=None,
    plot_for_pub=False,
    marker_size=3,
    category_alpha=None,
    shuffle=False,
    shuffle_seed=0,
    show_points=True,
    category_order=("LLAS", "CLAS", "BLAS"),
    kde=False,
    kde_levels=4,
    kde_gridsize=120,
    kde_smoothing=3.0,
    kde_fill=False,
    kde_linewidth=1.0,
    kde_level_min=0.1,
    kde_level_max=0.9,
    kde_log_levels=False,
):
    """Draw one recording's category-colored OFF scatter onto ``ax``.

    ``sub`` holds the OFFs for a single (subject, probe, structure). Applies the
    shared Cook's-distance influence filter (removes the top ``1 -
    cooks_percentile`` by Cook's D from BOTH the scatter and the fit), then draws
    each category in its palette color, optionally overlays an OLS regression
    line, and -- when ``gc_row`` (one row of ``compute_group_correlations``) is
    given -- annotates the Spearman rho. With ``log_x`` the leverage / fit / axis
    use ``log10(x_col)`` (non-positive ``x`` dropped). With ``plot_for_pub`` the
    rho annotation omits the "small/moderate/large" effect-size word.

    Point-cloud visibility knobs (no subsampling is ever applied):

    marker_size : float
        Base marker size, scaled by ``pp.scale``.
    category_alpha : dict, optional
        Per-category point alpha, e.g. ``{"LLAS": 0.2, "CLAS": 0.4, "BLAS": 0.6}``.
        Defaults to 0.3 for every category.
    shuffle : bool
        If True, all categories are drawn in one randomized-order scatter (seeded
        by ``shuffle_seed``) so no hue is systematically painted on top of
        another. If False, categories are drawn in ``category_order`` (last on top).
    show_points : bool
        If False, the raw points are omitted entirely (e.g. to show only the KDE
        contours).
    category_order : sequence of str
        Categories to draw and their back-to-front order (non-shuffle) / iteration
        order for the KDE contours. Defaults to the three tiers; pass e.g.
        ``("LLAS", "CLAS")`` when a merged two-category ``category`` column is used.
    kde : bool
        If True, overlay per-category 2D density contours. ``kde_*`` control the
        levels, grid resolution, Gaussian bandwidth in bins, fill vs. lines, line
        width, the lowest/highest contour elevation (``kde_level_min`` /
        ``kde_level_max`` as fractions of each category's peak density), and
        whether elevations are geometrically spaced (``kde_log_levels``). Lower
        ``kde_level_min`` to surface diffuse shoulders a peaked mode would
        otherwise clip. See :func:`_category_density_contours`.
    """
    valid = sub[[x_col, y_col]].dropna()
    if log_x:
        valid = valid[valid[x_col] > 0]
    keep_idx = valid.index
    if len(valid) > 2:
        xfit = np.log10(valid[x_col].values) if log_x else valid[x_col].values
        keep_mask = _influence_keep(
            xfit, valid[y_col].values, percentile=cooks_percentile
        )
        keep_idx = valid.index[keep_mask]
    sub = sub.loc[keep_idx]

    if category_alpha is None:
        category_alpha = {"LLAS": 0.3, "CLAS": 0.3, "BLAS": 0.3}

    if show_points and len(sub):
        if shuffle:
            # Single randomized-order scatter: bake per-category color+alpha into
            # an RGBA array (indexed by the column's own categories, so this works
            # for the 3-tier or a merged 2-category column), then permute the draw
            # order so occlusion is unbiased across hues.
            col_cats = list(sub["category"].cat.categories)
            codes = sub["category"].cat.codes.to_numpy()
            keep = codes >= 0
            xs = sub[x_col].to_numpy()[keep]
            ys = sub[y_col].to_numpy()[keep]
            codes = codes[keep]
            rgba_lookup = np.array(
                [to_rgba(colors[cat], category_alpha.get(cat, 0.3)) for cat in col_cats]
            )
            rgba = rgba_lookup[codes]
            order = np.random.default_rng(shuffle_seed).permutation(len(xs))
            ax.scatter(
                xs[order],
                ys[order],
                c=rgba[order],
                s=pp.scale(marker_size),
                rasterized=True,
            )
        else:
            for cat in category_order:
                cat_sub = sub[sub["category"] == cat]
                if cat_sub.empty:
                    continue
                ax.scatter(
                    cat_sub[x_col],
                    cat_sub[y_col],
                    color=colors[cat],
                    s=pp.scale(marker_size),
                    alpha=category_alpha.get(cat, 0.3),
                    label=cat,
                    rasterized=True,
                )

    if kde:
        _category_density_contours(
            ax,
            sub,
            x_col,
            y_col,
            colors,
            levels=kde_levels,
            gridsize=kde_gridsize,
            smoothing=kde_smoothing,
            fill=kde_fill,
            linewidths=kde_linewidth,
            level_min=kde_level_min,
            level_max=kde_level_max,
            log_levels=kde_log_levels,
            category_order=category_order,
        )

    if show_regline and len(sub) > 1:
        xfit = np.log10(sub[x_col].values) if log_x else sub[x_col].values
        coeffs = np.polyfit(xfit, sub[y_col].values, deg=1)
        xline = np.linspace(xfit.min(), xfit.max(), 100)
        xplot = 10**xline if log_x else xline
        ax.plot(xplot, np.polyval(coeffs, xline), color="k", lw=pp.scale(1))

    if log_x:
        ax.set_xscale("log")

    if gc_row is not None:
        effect_colors = {
            "negligible": "grey",
            "small": "#1a6fb5",
            "moderate": "#d4820a",
            "large": "#c0392b",
        }
        if plot_for_pub:
            text = f"rho={gc_row['rho']:.2f}"
            color = "black"
        else:
            text = f"rho={gc_row['rho']:.2f} ({gc_row['effect_label']})"
            color = effect_colors.get(gc_row["effect_label"], "black")
        ax.text(
            0.03,
            0.97,
            text,
            transform=ax.transAxes,
            va="top",
            color=color,
        )


def plot_offs_by_recording(
    offs,
    x_col,
    y_col,
    colors=None,
    ncols=5,
    show_regline=False,
    condition_filter="all",
    group_corrs=None,
    poster_mode=False,
    subject_map=None,
    cooks_percentile=0.999,
    log_x=False,
    suptitle=None,
):
    """Plot OFF properties by recording, optionally with correlation annotations.

    Parameters
    ----------
    condition_filter : str
        Which OFFs to include: "all", "NREM", or "Wake".
    group_corrs : pd.DataFrame, optional
        Output of ``correlation_stats.compute_group_correlations()``.
        If provided, each subplot is annotated with rho and effect label.
    poster_mode : bool
        If True, anonymize subject labels and omit probe from subplot titles.
    subject_map : dict, optional
        Mapping from real subject names to anonymized labels (e.g. "Subject1").
        Required when ``poster_mode`` is True.
    cooks_percentile : float
        Per-group influence cut: points whose Cook's distance (w.r.t. a simple
        OLS fit of ``y_col`` on ``x_col``) exceeds this quantile of the group's
        own D distribution are removed from BOTH the scatter and the regression
        fit. Default 0.995 (drop the top 0.5%).
    log_x : bool
        If True, leverage / Cook's distance and the regression line are computed
        on ``log10(x_col)`` instead of raw ``x_col``, and the x-axis is drawn on
        a log scale. This dissolves the skew-driven leverage of right-skewed OFF
        properties (area, span, duration). Non-positive ``x`` values are dropped.

    Returns
    -------
    matplotlib.figure.Figure
    """
    offs = _filter_by_condition(offs, condition_filter)

    if colors is None:
        colors = plots.get_category_palette()

    combos = (
        offs[["subject", "probe", "structure"]]
        .drop_duplicates()
        .sort_values(["subject", "probe", "structure"])
        .values
    )
    n = len(combos)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols, figsize=pp.scale(ncols * 6, nrows * 5), constrained_layout=True
    )
    axes_flat = axes.flatten()

    for i, (subj, prb, struct) in enumerate(combos):
        ax = axes_flat[i]
        mask = (
            (offs["subject"] == subj)
            & (offs["probe"] == prb)
            & (offs["structure"] == struct)
        )
        sub = offs.loc[mask]

        # Look up this group's precomputed correlation row (if any) so the shared
        # per-axis drawer can annotate rho.
        gc_row = None
        if group_corrs is not None:
            row = group_corrs[
                (group_corrs["subject"] == subj)
                & (group_corrs["probe"] == prb)
                & (group_corrs["structure"] == struct)
            ]
            if len(row) == 1:
                gc_row = row.iloc[0]

        # Cook's-distance influence filter, category scatter, regression line and
        # rho annotation are all handled by the shared drawer.
        _draw_recording_scatter(
            ax,
            sub,
            x_col,
            y_col,
            colors,
            show_regline=show_regline,
            cooks_percentile=cooks_percentile,
            log_x=log_x,
            gc_row=gc_row,
            plot_for_pub=False,
        )

        if poster_mode:
            ax.set_title(f"{subject_map[subj]}, {struct}")
        else:
            ax.set_title(f"{subj}\n{prb} {struct}")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.supxlabel(x_col)
    fig.supylabel(y_col)
    fig.suptitle(
        suptitle if suptitle is not None else f"condition_filter={condition_filter!r}"
    )

    plt.show()
    return fig


In [ ]:
xy_pairs = [
    ("area", "total_bipolar_inst_log_delta"),
    # ("area", "total_bipolar_inst_log_eta"),
    ("median_duration", "max_bipolar_inst_log_delta"),
    # ("median_duration", "max_bipolar_inst_log_eta"),
    ("span", "max_bipolar_inst_log_delta"),
    # ("span", "max_bipolar_inst_log_eta"),
    ("median_trace", "max_bipolar_inst_log_delta"),
    # ("median_trace", "max_bipolar_inst_log_eta"),
]
condition_filters = ["NREM", "Wake"]  # Also valid: "all"
group_cols = ["subject", "probe", "structure"]

# --- Scatter-plot recording restriction ------------------------------------
# The scatter plots below can get large (one subplot per (subject, probe,
# structure)). By default we only produce the scatter figures for a single
# recording, given as a list of (subject, structure) pairs. Set to None to
# scatter every recording (as the forest / meta-analysis cells always do).
SCATTER_RECORDING_FILTER = [("CNPIX12-Santiago", "mPPC")]


def _filter_scatter_recordings(offs, recording_filter):
    """Restrict OFFs to the (subject, structure) pairs in ``recording_filter``.

    Returns ``offs`` unchanged when ``recording_filter`` is None (plot every
    recording). Only affects the scatter cell; forest/meta cells are unchanged.
    """
    if recording_filter is None:
        return offs
    keep = {tuple(pair) for pair in recording_filter}
    mask = pd.Series(
        list(zip(offs["subject"], offs["structure"])), index=offs.index
    ).isin(keep)
    return offs[mask]


## Methodology

### Data source

This notebook deals only with whole-recording morphological OFFs
(`mua.files.get_full_offs_path`), with canonical postprocessing columns and bandpower
attached in-notebook. Each OFF is classified NREM vs Wake by its per-OFF `state` column,
so the NREM filter uses all NREM OFFs and the Wake filter uses all Wake OFFs across the
full recording. Neither is ever subset to the experimental condition windows.

### Category selections

`LLAS`/`CLAS`/`BLAS` are the nested OFF-property filter tiers
(`cnpix_local_sleep.off_tables`), assigned to each OFF as the most restrictive tier it
passes (`category`). Because the tiers nest (LLAS contains CLAS contains BLAS), each
family of plots (scatter and forest) is produced for each category selection in
`CATEGORY_SELECTIONS`:

- `LLAS-exclusive`: OFFs passing only the LLAS filters (`category == "LLAS"`)
- `CLAS-exclusive`: OFFs passing CLAS but not BLAS (`category == "CLAS"`)
- `BLAS-exclusive`: OFFs passing the BLAS filters (`category == "BLAS"`)
- `LLAS`: all OFFs, LLAS including CLAS and BLAS
- `CLAS`: CLAS including BLAS

### Problem

We want to assess whether OFF-period spatial and temporal properties (area, span, median
duration) are correlated with bandpower measures (delta, eta) during the OFF. Individual
OFF events are nested within (subject, probe, structure) recording groups, so
observations are not independent. With tens of thousands of OFFs per group, naive
correlation p-values are trivially small for any nonzero effect, which makes
significance testing on the full pooled sample uninformative.

### Approach

Correlation measure. Spearman rank correlation (rho) is the primary measure. OFF
properties (area, span, duration) are right-skewed, and Spearman is robust to monotonic
nonlinearity and distributional skew. 95% confidence intervals come from the Fisher
z-transform: `z = arctanh(rho)`, `SE(z) = 1/sqrt(N-3)`.

Per-group assessment. For each (subject, probe, structure) combo we report Spearman rho
with 95% CI and an effect-size classification (Cohen, 1988: `|rho| < 0.1` negligible,
`< 0.3` small, `< 0.5` moderate, `>= 0.5` large). Per-group p-values are not emphasized,
because at `N > 10k` they are uniformly near zero and carry no practical information
beyond the sign and magnitude of rho.

Across-group assessment. Each (subject, probe, structure) is treated as an independent
study, and the Fisher-z-transformed per-group correlations are pooled with a
DerSimonian-Laird random-effects meta-analysis (DerSimonian & Laird, 1986). This
accounts for varying group sizes via inverse-variance weighting, estimates between-group
heterogeneity (tau^2) and reports I^2 (the fraction of total variance attributable to
between-group differences), and yields a pooled rho with 95% CI and a z-test p-value for
H0: rho = 0. A forest plot shows per-group estimates with CIs and the overall pooled
diamond.

Multiple comparisons. Bonferroni correction is applied to the meta-analytic p-values
within each condition filter (alpha_corrected = 0.05 / k, where k is the number of x-y
pairs tested). Per-group annotations are descriptive and uncorrected. Comparisons across
condition filters and category selections are exploratory and uncorrected.

Sensitivity. Leave-one-out analysis re-runs the meta-analysis dropping each group in
turn, to check whether any single recording site disproportionately drives the overall
result.


## Scatter grid (LLAS, single recording)

A single 2x4 publication grid for the recording(s) in `SCATTER_RECORDING_FILTER`
(default CNPIX12-Santiago mPPC), LLAS only. Rows are the NREM (top) and Wake (bottom)
OFF populations; the four columns are, left to right, the `xy_pairs` (OFF property,
bandpower) relationships: `total ..._log_delta` vs `area`, `max ..._log_delta` vs
`median_duration`, `max ..._log_delta` vs `span`, and `max ..._log_delta` vs
`median_trace`. Each panel keeps the per-group Spearman rho annotation and OLS
regression line.

With `PLOT_FOR_PUB` the rho annotation drops the small/moderate/large effect-size word
and the y-axis (bandpower) labels are omitted, leaving a bare 7x3 inch SVG to be
labelled downstream, e.g. in Figma. Set `PLOT_FOR_PUB = False` to restore y labels,
condition row labels and a title while iterating.


In [ ]:
do_scatter_plots = True

# --- Point-cloud visibility knobs (tune here; no subsampling is ever applied) --
# Marker size (base value; scaled by pp.scale for the figma destination).
SCATTER_MARKER_SIZE = 0.2
# Per-category point alpha. Bump the tightest/rarest tiers (CLAS/BLAS) and/or
# fade the densest (LLAS) to pull each hue out of the pile.
SCATTER_CATEGORY_ALPHA = {
    "LLAS": 0.1,
    "CLAS": 0.3,
    "BLAS": 0.1,
}
# Draw all categories in one randomized-order scatter so no hue is systematically
# painted on top of another (fair occlusion). Set False to draw LLAS -> CLAS ->
# BLAS (BLAS on top), matching the original look.
SCATTER_SHUFFLE = True
SCATTER_SHUFFLE_SEED = 0

# Draw the raw points at all. Set False to show only the KDE contours below.
SCATTER_SHOW_POINTS = False

# Overlay a fast per-category 2D density estimate as contours (see
# _category_density_contours). Reads each category's shape without occlusion.
SCATTER_KDE = True
SCATTER_KDE_FILL = False  # filled bands vs. contour lines
SCATTER_KDE_LEVELS = 4  # number of contour levels
SCATTER_KDE_GRIDSIZE = 120  # histogram bins per axis (contour resolution)
SCATTER_KDE_SMOOTHING = 4.0  # Gaussian smoothing sigma in bins (~bandwidth)
SCATTER_KDE_LINEWIDTH = 1.0  # contour line width (base; scaled by pp.scale)
# Contour elevations as fractions of each category's peak density. Lower
# SCATTER_KDE_LEVEL_MIN to surface a diffuse shoulder that a sharply-peaked mode
# (e.g. the merged CLAS's dense BLAS spike in the span panel) would otherwise
# clip below the lowest ring; SCATTER_KDE_LOG_LEVELS spaces the rings
# geometrically across the density's dynamic range instead of bunching them near
# the peak (LEVEL_MIN must be > 0).
SCATTER_KDE_LEVEL_MIN = 0.05
SCATTER_KDE_LEVEL_MAX = 0.9
SCATTER_KDE_LOG_LEVELS = True

if do_scatter_plots:
    # Single-recording, LLAS-only publication scatter grid (2 rows x 4 cols):
    #   rows    = NREM (top), Wake (bottom)
    #   columns = the four (OFF property, bandpower) pairs in `xy_pairs`, so
    #             left-to-right: total delta vs area, max delta vs median
    #             duration, max delta vs span, max delta vs median trace.
    # The forest / meta-analysis cells below are unaffected (they still span
    # every recording).
    scatter_offs = _filter_scatter_recordings(offs, SCATTER_RECORDING_FILTER)
    scatter_offs = _filter_by_category_selection(scatter_offs, "LLAS")

    _rec = scatter_offs[["subject", "probe", "structure"]].drop_duplicates()
    if len(_rec) != 1:
        raise ValueError(
            "The scatter grid is a single-recording publication figure, but "
            f"SCATTER_RECORDING_FILTER selected {len(_rec)} recordings:\n{_rec}"
        )

    colors = plots.get_category_palette()
    scatter_conditions = ["NREM", "Wake"]  # grid rows

    with pp.destination("figma"):
        fig, axes = plt.subplots(
            len(scatter_conditions),
            len(xy_pairs),
            figsize=pp.scale(7, 3),
            constrained_layout=True,
            squeeze=False,
        )
        for r, cond in enumerate(scatter_conditions):
            cond_offs = _filter_by_condition(scatter_offs, cond)
            for c, (x_col, y_col) in enumerate(xy_pairs):
                ax = axes[r][c]
                gc = correlation_stats.compute_group_correlations(
                    cond_offs, x_col, y_col, group_cols
                )
                gc_row = gc.iloc[0] if len(gc) == 1 else None
                _draw_recording_scatter(
                    ax,
                    cond_offs,
                    x_col,
                    y_col,
                    colors,
                    show_regline=True,
                    gc_row=gc_row,
                    plot_for_pub=PLOT_FOR_PUB,
                    marker_size=SCATTER_MARKER_SIZE,
                    category_alpha=SCATTER_CATEGORY_ALPHA,
                    shuffle=SCATTER_SHUFFLE,
                    shuffle_seed=SCATTER_SHUFFLE_SEED,
                    show_points=SCATTER_SHOW_POINTS,
                    kde=SCATTER_KDE,
                    kde_levels=SCATTER_KDE_LEVELS,
                    kde_gridsize=SCATTER_KDE_GRIDSIZE,
                    kde_smoothing=SCATTER_KDE_SMOOTHING,
                    kde_fill=SCATTER_KDE_FILL,
                    kde_linewidth=SCATTER_KDE_LINEWIDTH,
                    kde_level_min=SCATTER_KDE_LEVEL_MIN,
                    kde_level_max=SCATTER_KDE_LEVEL_MAX,
                    kde_log_levels=SCATTER_KDE_LOG_LEVELS,
                )
                # Axis labels are dropped entirely for the publication figure
                # (bare 7x3 SVG, labelled downstream in Figma). While iterating
                # (PLOT_FOR_PUB=False): x labels on the bottom (Wake) row, y
                # labels (bandpower) on every panel.
                if not PLOT_FOR_PUB:
                    if r == len(scatter_conditions) - 1:
                        ax.set_xlabel(x_col)
                    ax.set_ylabel(y_col)

        if not PLOT_FOR_PUB:
            # Row (condition) labels + a title only while iterating. For the
            # publication SVG the grid is left bare for downstream labelling.
            for r, cond in enumerate(scatter_conditions):
                ax0 = axes[r][0]
                ax0.annotate(
                    cond,
                    xy=(0, 0.5),
                    xycoords=ax0.yaxis.label,
                    xytext=(-ax0.yaxis.labelpad - 20, 0),
                    textcoords="offset points",
                    ha="right",
                    va="center",
                    rotation=90,
                    fontweight="bold",
                )
            subj, prb, struct = _rec.iloc[0]
            fig.suptitle(f"LLAS scatter grid | {subj} {prb} {struct}")

        if save_plots:
            scatter_dir = OUTPUT_DIR / "LLAS" / "scatter"
            scatter_dir.mkdir(exist_ok=True, parents=True)
            fig.savefig(scatter_dir / "scatter_grid.svg")
        plt.show()


## Scatter grid (CLAS + BLAS merged, single recording)

The same 2x4 grid as above, but the CLAS-exclusive and BLAS OFFs are collapsed into a
single `CLAS` category, so only two categories are drawn: `LLAS` (green) and the merged
`CLAS` (orange). BLAS's own color, contours and point cloud are absorbed into CLAS,
leaving two colors, two contour sets and two point clouds per panel. Reuses the
`SCATTER_*` visibility knobs from the first scatter cell; per-panel rho reflects all
OFFs, since the merge only relabels hues.


In [ ]:
# Second scatter grid: CLAS-exclusive + BLAS OFFs collapsed into a single
# "CLAS" category, so only TWO categories are drawn -- LLAS (green) and the
# merged CLAS (orange); BLAS's own color / contours / cloud are absorbed into
# CLAS. Same single recording and same 2x4 layout as the first scatter cell,
# and it reuses every SCATTER_* visibility knob defined there (marker size,
# per-category alpha, shuffle, KDE).
do_scatter_plots_merged = True


def _merge_blas_into_clas(o):
    """Collapse the BLAS tier into CLAS, leaving two categories: LLAS, CLAS.

    Each OFF's exclusive tier label (LLAS / CLAS / BLAS) is remapped so BLAS ->
    CLAS; the result is a two-level ordered Categorical (LLAS, CLAS). "CLAS" then
    denotes the full CLAS tier (CLAS-exclusive union BLAS).
    """
    o = o.copy()
    merged = o["category"].astype("string").replace({"BLAS": "CLAS"})
    o["category"] = pd.Categorical(merged, categories=["LLAS", "CLAS"], ordered=True)
    return o


if do_scatter_plots_merged:
    merged_category_order = ["LLAS", "CLAS"]

    scatter_offs = _filter_scatter_recordings(offs, SCATTER_RECORDING_FILTER)
    scatter_offs = _filter_by_category_selection(scatter_offs, "LLAS")
    scatter_offs = _merge_blas_into_clas(scatter_offs)

    _rec = scatter_offs[["subject", "probe", "structure"]].drop_duplicates()
    if len(_rec) != 1:
        raise ValueError(
            "The scatter grid is a single-recording publication figure, but "
            f"SCATTER_RECORDING_FILTER selected {len(_rec)} recordings:\n{_rec}"
        )

    colors = plots.get_category_palette()
    scatter_conditions = ["NREM", "Wake"]  # grid rows

    with pp.destination("figma"):
        fig, axes = plt.subplots(
            len(scatter_conditions),
            len(xy_pairs),
            figsize=pp.scale(7, 3),
            constrained_layout=True,
            squeeze=False,
        )
        for r, cond in enumerate(scatter_conditions):
            cond_offs = _filter_by_condition(scatter_offs, cond)
            for c, (x_col, y_col) in enumerate(xy_pairs):
                ax = axes[r][c]
                gc = correlation_stats.compute_group_correlations(
                    cond_offs, x_col, y_col, group_cols
                )
                gc_row = gc.iloc[0] if len(gc) == 1 else None
                _draw_recording_scatter(
                    ax,
                    cond_offs,
                    x_col,
                    y_col,
                    colors,
                    show_regline=True,
                    gc_row=gc_row,
                    plot_for_pub=PLOT_FOR_PUB,
                    marker_size=SCATTER_MARKER_SIZE,
                    category_alpha=SCATTER_CATEGORY_ALPHA,
                    shuffle=SCATTER_SHUFFLE,
                    shuffle_seed=SCATTER_SHUFFLE_SEED,
                    show_points=SCATTER_SHOW_POINTS,
                    category_order=merged_category_order,
                    kde=SCATTER_KDE,
                    kde_levels=SCATTER_KDE_LEVELS,
                    kde_gridsize=SCATTER_KDE_GRIDSIZE,
                    kde_smoothing=SCATTER_KDE_SMOOTHING,
                    kde_fill=SCATTER_KDE_FILL,
                    kde_linewidth=SCATTER_KDE_LINEWIDTH,
                    kde_level_min=SCATTER_KDE_LEVEL_MIN,
                    kde_level_max=SCATTER_KDE_LEVEL_MAX,
                    kde_log_levels=SCATTER_KDE_LOG_LEVELS,
                )
                # Axis labels dropped entirely for the publication figure; while
                # iterating (PLOT_FOR_PUB=False): x labels on the bottom (Wake)
                # row, y labels (bandpower) on every panel.
                if not PLOT_FOR_PUB:
                    if r == len(scatter_conditions) - 1:
                        ax.set_xlabel(x_col)
                    ax.set_ylabel(y_col)

        if not PLOT_FOR_PUB:
            for r, cond in enumerate(scatter_conditions):
                ax0 = axes[r][0]
                ax0.annotate(
                    cond,
                    xy=(0, 0.5),
                    xycoords=ax0.yaxis.label,
                    xytext=(-ax0.yaxis.labelpad - 20, 0),
                    textcoords="offset points",
                    ha="right",
                    va="center",
                    rotation=90,
                    fontweight="bold",
                )
            subj, prb, struct = _rec.iloc[0]
            fig.suptitle(f"CLAS+BLAS-merged scatter grid | {subj} {prb} {struct}")

        if save_plots:
            scatter_dir = OUTPUT_DIR / "clas_blas_merged" / "scatter"
            scatter_dir.mkdir(exist_ok=True, parents=True)
            fig.savefig(scatter_dir / "scatter_grid.svg")
        plt.show()


## Per-group correlations and meta-analysis

For each condition filter and x-y pair:
- Compute per-group Spearman rho with 95% CIs
- Run DerSimonian-Laird random-effects meta-analysis across groups
- Display forest plot and summary

In [ ]:
def plot_forest(
    group_corrs: pd.DataFrame,
    meta: dict,
    title: str = "",
    ax: plt.Axes | None = None,
    plot_for_poster: bool = False,
    color: str = "steelblue",
    elinewidth=1,
    markersize=1,
    capsize=1,
) -> plt.Axes:
    """Forest plot of per-group correlations with pooled estimate.

    Parameters
    ----------
    group_corrs
        Output of :func:`compute_group_correlations`.
    meta
        Output of :func:`meta_analyze_correlations`.
    title
        Plot title.
    ax
        Matplotlib axes to draw on.  Created if *None*.
    color
        Color of the per-group markers and their CI error bars. Defaults to
        ``"steelblue"`` (unchanged from the original single-color forests).
        The pooled meta-analytic diamond and the ``rho = 0`` reference line
        are left fixed so a multi-color grid stays legible.
    """
    id_cols = [c for c in ("subject", "probe", "structure") if c in group_corrs.columns]
    labels = [
        " / ".join(str(row[c]) for c in id_cols) for _, row in group_corrs.iterrows()
    ]

    k = len(group_corrs)
    if ax is None:
        fig_width = 2.0 if plot_for_poster else 4.0
        fig_height = max(4, 0.15 * (k + 2))
        _, ax = plt.subplots(
            figsize=pp.scale(fig_width, fig_height), constrained_layout=True
        )

    y_positions = np.arange(k)[::-1]

    # Per-group CIs
    rhos = group_corrs["rho"].values
    ci_los = group_corrs["ci_lo"].values
    ci_his = group_corrs["ci_hi"].values
    ax.errorbar(
        rhos,
        y_positions,
        xerr=[rhos - ci_los, ci_his - rhos],
        fmt="o",
        color=color,
        ecolor=color,
        elinewidth=pp.scale(elinewidth),
        markersize=pp.scale(markersize),
        capsize=pp.scale(capsize),
    )

    # Overall meta-analytic estimate (diamond)
    y_overall = -1.5
    diamond_hw = 0.4  # half-width in y
    diamond_x = [
        meta["ci_lo"],
        meta["overall_rho"],
        meta["ci_hi"],
        meta["overall_rho"],
    ]
    diamond_y = [
        y_overall,
        y_overall + diamond_hw,
        y_overall,
        y_overall - diamond_hw,
    ]
    ax.fill(diamond_x, diamond_y, color="firebrick", alpha=0.7)

    # Reference line at rho = 0
    ax.axvline(0, color="grey", linestyle="--", linewidth=pp.scale(0.8))

    # Labels
    if plot_for_poster:
        ax.set_yticks([])
        ax.set_yticklabels([])
    else:
        ax.set_yticks(list(y_positions) + [y_overall])
        ax.set_yticklabels(labels + ["Overall"])
        ax.set_xlabel("Spearman rho")
        if title:
            ax.set_title(title)

        # Right-side annotations: rho [CI]
        for i, (rho, lo, hi, n) in enumerate(
            zip(rhos, ci_los, ci_his, group_corrs["n"].values)
        ):
            ax.text(
                1.0,
                y_positions[i],
                f" {rho:+.3f} [{lo:+.3f}, {hi:+.3f}]  N={n}",
                transform=ax.get_yaxis_transform(),
                va="center",
                family="monospace",
            )
        ax.text(
            1.0,
            y_overall,
            f" {meta['overall_rho']:+.3f} [{meta['ci_lo']:+.3f}, {meta['ci_hi']:+.3f}]",
            transform=ax.get_yaxis_transform(),
            va="center",
            fontweight="bold",
            family="monospace",
        )

    if meta["overall_rho"] < 0:
        ax.set_xlim(-1.0, 0.05)
        ax.set_xticks([-1.0, -0.5, 0.0])
    else:
        ax.set_xlim(-0.05, 1.0)
        ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_ylim(y_overall - 1, y_positions[0] + 1)
    return ax


In [ ]:
show_plots = False
show_tables = False
all_meta_results = []
display_group_cols = [c for c in group_cols if not (plot_for_poster and c == "probe")]

with pp.destination("figma"):
    for selection in CATEGORY_SELECTIONS:
        sel_offs = _filter_by_category_selection(offs, selection)
        forest_dir = OUTPUT_DIR / selection / "forest"
        forest_dir.mkdir(exist_ok=True, parents=True)
        for cond in condition_filters:
            cond_offs = _filter_by_condition(sel_offs, cond)
            for x_col, y_col in xy_pairs:
                gc = correlation_stats.compute_group_correlations(
                    cond_offs, x_col, y_col, group_cols
                )
                gc_display = _anonymize_gc(gc, subject_map) if plot_for_poster else gc
                meta = correlation_stats.meta_analyze_correlations(gc_display)

                # Print summary
                print("=" * 60)
                print(f"Selection: {selection}  |  Condition: {cond}")
                print(correlation_stats.format_meta_summary(meta, x_col, y_col))
                print()

                # Forest plot
                ax = plot_forest(
                    gc_display,
                    meta,
                    title=f"{selection} | {x_col} vs {y_col}  (condition={cond})",
                    plot_for_poster=plot_for_poster,
                )
                if save_plots:
                    ax.figure.savefig(forest_dir / f"{cond}_{x_col}_vs_{y_col}.svg")
                if show_plots:
                    plt.show()
                else:
                    plt.close(ax.figure)

                if show_tables:
                    # Per-group table
                    display(
                        gc_display[
                            display_group_cols
                            + ["rho", "ci_lo", "ci_hi", "n", "effect_label"]
                        ]
                        .sort_values("rho", ascending=False)
                        .reset_index(drop=True)
                        .style.format(
                            {"rho": "{:.3f}", "ci_lo": "{:.3f}", "ci_hi": "{:.3f}"}
                        )
                    )

                # Store for summary table
                all_meta_results.append(
                    dict(
                        selection=selection,
                        condition=cond,
                        x_col=x_col,
                        y_col=y_col,
                        **meta,
                    )
                )


## Summary table

Overall meta-analytic results for all selection x condition x pair combinations. Bonferroni-corrected significance threshold: alpha = 0.05 / (number of x-y pairs) per condition.

In [ ]:
summary = pd.DataFrame(all_meta_results)
summary["CI"] = summary.apply(lambda r: f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1)

n_pairs = len(xy_pairs)
alpha_corrected = 0.05 / n_pairs
summary["sig_bonf"] = summary["p_value"] < alpha_corrected
print(f"Bonferroni threshold: alpha = 0.05 / {n_pairs} = {alpha_corrected:.4f}")

display(
    summary[
        [
            "selection",
            "condition",
            "x_col",
            "y_col",
            "overall_rho",
            "CI",
            "p_value",
            "i_squared",
            "q_pvalue",
            "k",
            "effect_label",
            "sig_bonf",
        ]
    ]
    .style.format(
        {
            "overall_rho": "{:.3f}",
            "p_value": "{:.2e}",
            "i_squared": "{:.1f}%",
            "q_pvalue": "{:.2e}",
        }
    )
    .set_caption("Random-effects meta-analysis of Spearman correlations")
)


## Leave-one-out sensitivity (condition="all", per selection)

Check whether any single group disproportionately drives the overall result, for each category selection.

In [ ]:
for selection in CATEGORY_SELECTIONS:
    sel_offs = _filter_by_category_selection(offs, selection)
    print(f"\n{'=' * 60}\nSelection: {selection}")
    for x_col, y_col in xy_pairs:
        gc = correlation_stats.compute_group_correlations(
            sel_offs, x_col, y_col, group_cols
        )
        gc_display = _anonymize_gc(gc, subject_map) if plot_for_poster else gc
        loo = correlation_stats.leave_one_out(gc_display)
        meta = correlation_stats.meta_analyze_correlations(gc_display)

        print(f"\n{x_col} vs {y_col}")
        print(f"  Full: rho = {meta['overall_rho']:.3f}")
        print(
            f"  LOO range: [{loo['overall_rho'].min():.3f}, {loo['overall_rho'].max():.3f}]"
        )
        shift = (loo["overall_rho"] - meta["overall_rho"]).abs()
        if shift.max() > 0.02:
            worst = loo.loc[shift.idxmax()]
            print(
                f"  Largest shift: dropping {worst['dropped']}"
                f" -> rho = {worst['overall_rho']:.3f}"
                f" (delta = {shift.max():+.3f})"
            )
        else:
            print("  No group shifts overall rho by more than 0.02")


## Combined forest grid

All forest plots in one figure: each row is a category selection, the first
four columns are the NREM forests (one per x-y pair) and the last four are
the Wake forests.

In [ ]:
# Big grid of every forest plot: one row per category selection, with the four
# NREM forests (one per x-y pair) in the first columns and the four Wake forests
# in the last columns. Sizes are wrapped in `pp.scale` so they track the pubplots
# destination; font sizes are left entirely to the pubplots rcParams.
grid_conditions = ["NREM", "Wake"]  # column blocks, in order
n_pairs = len(xy_pairs)
n_rows = len(CATEGORY_SELECTIONS)
n_cols = len(grid_conditions) * n_pairs

with pp.destination("figma"):
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=pp.scale(n_cols * 2.0, n_rows * 2.4),
        constrained_layout=True,
        squeeze=False,
    )

    for r, selection in enumerate(CATEGORY_SELECTIONS):
        sel_offs = _filter_by_category_selection(offs, selection)
        for ci, cond in enumerate(grid_conditions):
            cond_offs = _filter_by_condition(sel_offs, cond)
            for pj, (x_col, y_col) in enumerate(xy_pairs):
                col = ci * n_pairs + pj
                ax = axes[r][col]

                gc = correlation_stats.compute_group_correlations(
                    cond_offs, x_col, y_col, group_cols
                )
                gc_display = _anonymize_gc(gc, subject_map) if plot_for_poster else gc
                meta = correlation_stats.meta_analyze_correlations(gc_display)

                # Compact (poster) styling so the right-side per-group text
                # doesn't collide across the tight grid.
                plot_forest(gc_display, meta, ax=ax, plot_for_poster=True)

                # Column headers (condition + x-y pair) only on the top row.
                if r == 0:
                    ax.set_title(f"{cond}\n{x_col}\nvs {y_col}")
                # Row labels (selection) only on the first column.
                if col == 0:
                    ax.set_ylabel(selection)

    fig.suptitle("Forest plots: row = selection, cols 1-4 = NREM, cols 5-8 = Wake")

    if save_plots:
        fig.savefig(OUTPUT_DIR / "forest_grid.svg")
    plt.show()


## Three-row forest grid (LLAS / LLAS-exclusive / CLAS)

Publication variant of the combined forest grid restricted to three category selections (top-to-bottom: LLAS, LLAS-exclusive, CLAS), 7" wide, no figure title. Per-group markers + error bars are colored per row: LLAS in black, LLAS-exclusive in the LLAS scatter color, CLAS in the CLAS scatter color.

In [ ]:
# Three-row publication forest grid. Rows (top -> bottom) are the LLAS,
# LLAS-exclusive, and CLAS category selections. Per-group markers + error bars
# are colored per row: LLAS -> black, LLAS-exclusive -> the LLAS scatter color,
# CLAS -> the CLAS scatter color (the pooled diamond stays firebrick). Total
# figure width is fixed at 7"; no figure title. Sizes are wrapped in `pp.scale`
# so they track the pubplots destination.
FOREST3_SELECTIONS = ["LLAS", "LLAS-exclusive", "CLAS"]  # top -> bottom
_scatter_palette = plots.get_category_palette()
FOREST3_ROW_COLORS = {
    "LLAS": "black",
    "LLAS-exclusive": _scatter_palette["LLAS"],  # green (LLAS scatter tier)
    "CLAS": _scatter_palette["CLAS"],  # orange (CLAS scatter tier)
}
FOREST3_FIG_WIDTH = 5.6  # inches, full figure width
FOREST3_HEIGHT_PER_ROW = 1.6  # inches per selection row (tune to taste)

_f3_conditions = ["NREM", "Wake"]  # column blocks, in order
_f3_n_pairs = len(xy_pairs)
_f3_n_rows = len(FOREST3_SELECTIONS)
_f3_n_cols = len(_f3_conditions) * _f3_n_pairs

with pp.destination("figma"):
    fig, axes = plt.subplots(
        _f3_n_rows,
        _f3_n_cols,
        figsize=pp.scale(FOREST3_FIG_WIDTH, _f3_n_rows * FOREST3_HEIGHT_PER_ROW),
        constrained_layout=True,
        squeeze=False,
    )

    for r, selection in enumerate(FOREST3_SELECTIONS):
        sel_offs = _filter_by_category_selection(offs, selection)
        row_color = FOREST3_ROW_COLORS[selection]
        for ci, cond in enumerate(_f3_conditions):
            cond_offs = _filter_by_condition(sel_offs, cond)
            for pj, (x_col, y_col) in enumerate(xy_pairs):
                col = ci * _f3_n_pairs + pj
                ax = axes[r][col]

                gc = correlation_stats.compute_group_correlations(
                    cond_offs, x_col, y_col, group_cols
                )
                gc_display = _anonymize_gc(gc, subject_map) if plot_for_poster else gc
                meta = correlation_stats.meta_analyze_correlations(gc_display)

                # Compact (poster) styling hides the crowded per-group y labels.
                plot_forest(
                    gc_display,
                    meta,
                    ax=ax,
                    plot_for_poster=PLOT_FOR_PUB,
                    color=row_color,
                )

                # Column headers (condition + x-y pair) only on the top row.
                if not PLOT_FOR_PUB and r == 0:
                    ax.set_title(f"{cond}\n{x_col}\nvs {y_col}")
                # Row labels (selection) only on the first column.
                if col == 0:
                    ax.set_ylabel(selection)

    # No figure title (per request).

    if save_plots:
        fig.savefig(OUTPUT_DIR / "forest_grid_llas_llasexcl_clas.svg")
    plt.show()


## Single-axes forest grid (three OFF categories overlaid, recording-aligned)

A collapsed variant of the three-row grid above: for each (condition x metric-pair)
column the LLAS / LLAS-exclusive / CLAS forests are drawn on one shared axes instead of
three stacked rows. Each recording occupies one vertical slot; the three categories are
drawn at a small vertical offset within the slot so their points and error bars do not
collide, and the slot pitch keeps each recording's triplet grouped and separated from
the next, so a vertical position still maps to a single recording. All 29 recordings are
present in all three categories, in every panel. Category colors are LLAS black,
LLAS-exclusive green, CLAS orange; the three meta-analytic pooled diamonds are stacked
below the band in the same colors, with the category names labelling them in the first
column. Same width, no figure title.


In [ ]:
# Collapsed single-row forest grid, RECORDING-ALIGNED. The three category forests
# share ONE axes per column. Within each recording's slot the categories get a small
# vertical offset (so their markers/error bars don't collide) and the slot pitch
# keeps each recording's triplet grouped and separated from the next -- so a vertical
# position still maps to one recording (here LLAS/LLAS-exclusive/CLAS are present in
# all 29 recordings, in every panel). Pooled diamonds are stacked below the band,
# category-colored; category names label the diamonds in the first column. Sizes
# wrapped in pp.scale so they track the pubplots destination.
FOREST1_FIG_WIDTH = 6  # inches, full figure width
FOREST1_ROW_HEIGHT = 3  # inches, single row (taller than a flat overlay so the
# offset triplets have vertical room)
FOREST1_ROW_PITCH = 5  # y-distance between consecutive recordings (data coords)
FOREST1_CAT_DY = 0.62  # per-category vertical offset step within a slot
FOREST1_DIAMOND_TOP = -1.6 * FOREST1_ROW_PITCH  # y of the top pooled diamond
FOREST1_DIAMOND_SPACING = 3.25  # y-gap between stacked diamonds
FOREST1_HW = 0.45 * FOREST1_ROW_PITCH  # diamond half-height (y)


def _forest_overlay(ax, entries, first_col):
    """Recording-aligned overlay of several category forests on one axes.

    ``entries`` is a list of ``(name, gc, meta, color)``. The recordings (shared
    ``group_cols`` key) are ordered once; each occupies a slot ``FOREST1_ROW_PITCH``
    apart, and the categories are drawn at symmetric ``FOREST1_CAT_DY`` offsets within
    the slot so their markers/error bars separate while a recording's triplet stays
    grouped. A category contributes a marker only for recordings it actually has
    (outer-join -- no row is dropped because one category is missing it). Each
    category's pooled diamond is stacked below the band in its own color; the first
    column labels the diamonds with the category names.
    """
    # Shared recording order (union across categories, group-key sorted).
    recs, seen = [], set()
    for _, gc, _, _ in entries:
        for key in gc[group_cols].itertuples(index=False, name=None):
            if key not in seen:
                seen.add(key)
                recs.append(key)
    recs.sort()
    rank = {k: i for i, k in enumerate(recs)}
    R = max(1, len(recs))
    n_cat = len(entries)
    base = np.arange(R)[::-1] * FOREST1_ROW_PITCH  # first recording at top
    offsets = (np.arange(n_cat) - (n_cat - 1) / 2.0) * FOREST1_CAT_DY

    lo_vals, hi_vals, diamond_ys = [], [], []
    for i, (name, gc, meta, color) in enumerate(entries):
        keys = [tuple(t) for t in gc[group_cols].itertuples(index=False, name=None)]
        yy = np.array([base[rank[k]] for k in keys]) + offsets[i]
        rho = gc["rho"].to_numpy()
        clo = gc["ci_lo"].to_numpy()
        chi = gc["ci_hi"].to_numpy()
        ax.errorbar(
            rho,
            yy,
            xerr=[rho - clo, chi - rho],
            fmt="o",
            color=color,
            ecolor=color,
            elinewidth=pp.scale(0.5),
            markersize=pp.scale(1.4),
            capsize=0,
            alpha=0.75,
            zorder=2,
        )
        yd = FOREST1_DIAMOND_TOP - i * FOREST1_DIAMOND_SPACING
        diamond_ys.append(yd)
        ax.fill(
            [meta["ci_lo"], meta["overall_rho"], meta["ci_hi"], meta["overall_rho"]],
            [yd, yd + FOREST1_HW, yd, yd - FOREST1_HW],
            color=color,
            alpha=0.85,
            zorder=5,
        )
        lo_vals += [float(clo.min()), meta["ci_lo"]]
        hi_vals += [float(chi.max()), meta["ci_hi"]]

    ax.axvline(0, color="grey", ls="--", lw=pp.scale(0.8), zorder=0)
    lo = max(-1.02, min(0.0, min(lo_vals)) - 0.05)
    hi = min(1.02, max(0.0, max(hi_vals)) + 0.05)
    ax.set_xlim(lo, hi)
    ax.set_xticks(
        [t for t in (-1.0, -0.5, 0.0, 0.5, 1.0) if lo - 1e-9 <= t <= hi + 1e-9]
    )
    ax.set_ylim(
        min(diamond_ys) - FOREST1_HW - FOREST1_ROW_PITCH, base[0] + FOREST1_ROW_PITCH
    )
    if first_col:
        ax.set_yticks(diamond_ys)
        ax.set_yticklabels([name for name, *_ in entries])
        for tl, (_n, _g, _m, c) in zip(ax.get_yticklabels(), entries):
            tl.set_color(c)
    else:
        ax.set_yticks([])
    return ax


with pp.destination("figma"):
    fig, axes = plt.subplots(
        1,
        _f3_n_cols,
        figsize=pp.scale(FOREST1_FIG_WIDTH, FOREST1_ROW_HEIGHT),
        constrained_layout=True,
        squeeze=False,
    )
    _f1_sel_offs = {
        s: _filter_by_category_selection(offs, s) for s in FOREST3_SELECTIONS
    }
    for ci, cond in enumerate(_f3_conditions):
        for pj, (x_col, y_col) in enumerate(xy_pairs):
            col = ci * _f3_n_pairs + pj
            ax = axes[0][col]
            entries = []
            for selection in FOREST3_SELECTIONS:
                cond_offs = _filter_by_condition(_f1_sel_offs[selection], cond)
                gc = correlation_stats.compute_group_correlations(
                    cond_offs, x_col, y_col, group_cols
                )
                # Key/align on the raw gc (keeps "probe"); anonymization is a no-op
                # here since this overlay shows no per-recording labels.
                if len(gc) == 0:
                    continue
                meta = correlation_stats.meta_analyze_correlations(gc)
                entries.append((selection, gc, meta, FOREST3_ROW_COLORS[selection]))
            if not entries:
                continue
            _forest_overlay(ax, entries, first_col=(col == 0))
            # Column headers only on the quick-look (non-pub) figure.
            if not PLOT_FOR_PUB:
                ax.set_title(f"{cond}\n{x_col}\nvs {y_col}")

    # No figure title (per request).
    if save_plots:
        fig.savefig(OUTPUT_DIR / "forest_grid_overlaid_llas_llasexcl_clas.svg")
    plt.show()
